In [ ]:
# Abira Demello - Statistical Analysis

In [1]:
import pandas as pd 
import polars as pl

# MSTL - Multi Seasonal Trend decompostion using LOESS
from statsmodels.tsa.seasonal import MSTL
import warnings

In [2]:
from tqdm import tqdm # shows a progress bar when looping through data; Useful when processing many time series
tqdm.pandas()

In [3]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [10]:
Data = pd.read_parquet('labor-force-data-set.parquet')

In [13]:
Data

,ProvinceID,NOC_GroupingID,datestamp,Full-time employment,y,unique_id,month,quarter,month_in_quarter,month_in_quarter_sin,month_in_quarter_cos,month_sin,month_cos,quarter_sin,quarter_cos,week,week_sin,week_cos
0,10.0,1.0,2000-01-01,151.6,151.6,547,1,1,1,8.660254e-01,-0.5,5.000000e-01,8.660254e-01,1.000000e+00,6.123234e-17,52,6.432491e-16,1.000000e+00
1,10.0,1.0,2000-02-01,152.1,152.1,547,2,1,2,-8.660254e-01,-0.5,8.660254e-01,5.000000e-01,1.000000e+00,6.123234e-17,5,5.680647e-01,8.229839e-01
2,10.0,1.0,2000-03-01,152.5,152.5,547,3,1,3,-2.449294e-16,1.0,1.000000e+00,6.123234e-17,1.000000e+00,6.123234e-17,9,8.854560e-01,4.647232e-01
3,10.0,1.0,2000-04-01,156.7,156.7,547,4,2,1,8.660254e-01,-0.5,8.660254e-01,-5.000000e-01,1.224647e-16,-1.000000e+00,13,1.000000e+00,-1.608123e-16
4,10.0,1.0,2000-05-01,170.9,170.9,547,5,2,2,-8.660254e-01,-0.5,5.000000e-01,-8.660254e-01,1.224647e-16,-1.000000e+00,18,8.229839e-01,-5.680647e-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
171181,59.0,59.0,2025-05-01,18.3,18.3,161,5,2,2,-8.660254e-01,-0.5,5.000000e-01,-8.660254e-01,1.224647e-16,-1.000000e+00,18,8.229839e-01,-5.680647e-01
171182,59.0,59.0,2025-06-01,17.0,17.0,161,6,2,3,-2.449294e-16,1.0,1.224647e-16,-1.000000e+00,1.224647e-16,-1.000000e+00,22,4.647232e-01,-8.854560e-01
171183,59.0,59.0,2025-07-01,15.6,15.6,161,7,3,1,8.660254e-01,-0.5,-5.000000e-01,-8.660254e-01,-1.000000e+00,-1.836970e-16,27,-1.205367e-01,-9.927089e-01
171184,59.0,59.0,2025-08-01,12.6,12.6,161,8,3,2,-8.660254e-01,-0.5,-8.660254e-01,-5.000000e-01,-1.000000e+00,-1.836970e-16,31,-5.680647e-01,-8.229839e-01


In [14]:
# Slimming down the dataset

DataReduced = Data[['unique_id','NOC_GroupingID','ProvinceID','datestamp','y']].dropna()
DataReduced.head(10)

,unique_id,NOC_GroupingID,ProvinceID,datestamp,y
0,547,1.0,10.0,2000-01-01,151.6
1,547,1.0,10.0,2000-02-01,152.1
2,547,1.0,10.0,2000-03-01,152.5
3,547,1.0,10.0,2000-04-01,156.7
4,547,1.0,10.0,2000-05-01,170.9
5,547,1.0,10.0,2000-06-01,179.2
6,547,1.0,10.0,2000-07-01,192.3
7,547,1.0,10.0,2000-08-01,188.7
8,547,1.0,10.0,2000-09-01,179.0
9,547,1.0,10.0,2000-10-01,172.1


In [15]:
# Converting the pandas DF into a Polars DF 
# Polars is significantly faster at filtering and slicing large datasets
df = pl.from_pandas(DataReduced)

In [16]:
del DataReduced

Seasonal Decomposition 

In [19]:
#Sample: "Professional occupations in engineering [213]" in Ontario 

Sample = df.filter((pl.col('ProvinceID')==35) & (pl.col('NOC_GroupingID')==18)).select(['datestamp','y']).sort(by=['datestamp'], descending=False).to_pandas()
Sample

,datestamp,y
0,2000-01-01,66.9
1,2000-02-01,71.5
2,2000-03-01,78.1
3,2000-04-01,68.5
4,2000-05-01,64.7
...,...,...
304,2025-05-01,98.1
305,2025-06-01,98.1
306,2025-07-01,110.0
307,2025-08-01,105.5


In [20]:
# Converts the date column into a proper datetime format so pandas and MSTL understand it as time, not just text.
Sample['datestamp'] = pd.to_datetime(Sample['datestamp'])

In [21]:
Sample.dtypes

datestamp    datetime64[ns]
y                   float64
dtype: object

In [23]:
# this reshapes the data into a proper time series format - a single column of employment numbers where the row labels are the dates (YYYY-MM-DD) 
# MSTL requires this format to understand the time ordering

SampleTS = Sample.sort_values('datestamp')['y']
SampleTS.index = Sample['datestamp']
SampleTS

datestamp
2000-01-01     66.9
2000-02-01     71.5
2000-03-01     78.1
2000-04-01     68.5
2000-05-01     64.7
              ...  
2025-05-01     98.1
2025-06-01     98.1
2025-07-01    110.0
2025-08-01    105.5
2025-09-01    102.4
Name: y, Length: 309, dtype: float64

In [24]:
# This is the decomposition. we're tellig MSTL to break this time series into components by looking for 3 different seasonal cycle lengths
# 12 - annual seasonality 
# 6 - semi annual 
# 3 - quarterly (patterns repeating every 3 months)
mstl = MSTL(SampleTS, periods=[12, 6, 3])
result = mstl.fit()

In [25]:
# MSTL splits the original series into 4 pieces
# observed: the original raw employment numbers
observed = result.observed

# trend: the smooth long-term direction from 2000 to 2025
trend = result.trend

# the 3 seasonal patterns (3,6,12 months) - the repeating wiggles at each cycle length
seasonalities = result.seasonal

# residual : the leftover noise that none of the above couple explain
resid = result.resid

In [26]:
df_decomp = (
    observed.rename("observed")
    .to_frame()
    .join(trend.rename("trend"))
    .join(resid.rename("resid"))
    .join(seasonalities)  # already a DataFrame
)

In [27]:
# Copy the datetime index into its own column 'ds' for explicit date access during downstream processing
df_decomp['ds'] = df_decomp.index.values
df_decomp


,observed,trend,resid,seasonal_3,seasonal_6,seasonal_12,ds
datestamp,,,,,,,
2000-01-01,66.9,69.593255,0.537614,-1.746469,-1.562143,0.077743,2000-01-01
2000-02-01,71.5,69.765834,-0.531222,0.182931,0.138196,1.944261,2000-02-01
2000-03-01,78.1,69.953260,4.448634,1.398215,2.399446,-0.099556,2000-03-01
2000-04-01,68.5,70.156885,1.065773,-1.556273,1.914154,-3.080539,2000-04-01
2000-05-01,64.7,70.374648,-1.521208,0.264579,-0.387220,-4.030798,2000-05-01
...,...,...,...,...,...,...,...
2025-05-01,98.1,106.931901,-7.435525,-2.351277,4.079785,-3.124883,2025-05-01
2025-06-01,98.1,106.127545,-6.124126,-1.523521,0.407496,-0.787393,2025-06-01
2025-07-01,110.0,105.304168,4.738475,4.338528,-1.819512,-2.561658,2025-07-01


In [28]:
df_decomp = df_decomp.reset_index(drop=True)

In [29]:
df_decomp.dtypes

observed              float64
trend                 float64
resid                 float64
seasonal_3            float64
seasonal_6            float64
seasonal_12           float64
ds             datetime64[ns]
dtype: object

In [35]:
# creates a 6-panel stacked chart, all sharing the time axis on the x-axis
# each panel gets one component of the decomposition 

fig = make_subplots(
    rows=6,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.05,
    subplot_titles=[
        "Employment in Thousands",
        "Trend",
        "Annual Seasonality",
        "Semi-Annual Seasonality",
        "Quarterly Seasonality",
        "Residuals"
        
    ]
)

# Observed: solid line; raw data - shown as-is
fig.add_trace(
    go.Scatter(
        x=df_decomp['ds'],
        y=df_decomp['observed'],
        mode="lines",
        name="Employment in Thousands",
        line=dict(color='#13256B', width=2.5)
    ),
    row=1,
    col=1
)

# Trend: dashed line; same color as observed but dashed to signal its a smoothed version
fig.add_trace(
    go.Scatter(
        x=df_decomp['ds'],
        y=df_decomp['trend'],
        mode="lines",
        name="Trend",
        line=dict(color='#13256B', width=2.5, dash='dash')
    ),
    row=2,
    col=1
)

# Seasonal_12: solid line ; annual cycle
fig.add_trace(
    go.Scatter(
        x=df_decomp['ds'],
        y=df_decomp['seasonal_12'],
        mode="lines",
        name="Annual Seasonality",
        line=dict(color='#064241', width=2.5)
    ),
    row=3,
    col=1
)

# Seasonal_6: semi annual
fig.add_trace(
    go.Scatter(
        x=df_decomp['ds'],
        y=df_decomp['seasonal_6'],
        mode="lines",
        name="Semi-Annual Seasonality",
        line=dict(color='#17615E', width=2.5)
    ),
    row=4,
    col=1
)

#Seasonal_3: quarterly cycle
fig.add_trace(
    go.Scatter(
        x=df_decomp['ds'],
        y=df_decomp['seasonal_3'],
        mode="lines",
        name="Quarterly Seasonality",
        line=dict(color="#42B3AF", width=2.5)
    ),
    row=5,
    col=1
)

# Residuals: dots; Residuals are shown as scatter dots — not a line — because they're random noise with no meaningful connection between points
fig.add_trace(
    go.Scatter(
        x=df_decomp['ds'],
        y=df_decomp['resid'],
        mode="markers",
        name="Residuals",
        marker=dict(color="#580A0A", size=8)
    ),
    row=6,
    col=1
)



fig.update_layout(
    height=900,
    width=1500,
    title="Multi Seasonal Decomposition of Professional occupations in engineering [213] in Ontario",
    showlegend=False
)
fig.update_layout(template='ggplot2')
fig.show()

In [36]:
fig.write_image('figures/seasonal_decomp_Sample.svg')

In [41]:
Unique_Ids_df = df.select(['unique_id','ProvinceID','NOC_GroupingID']).unique().to_pandas().sort_values(by='unique_id')
Unique_Ids_df

,unique_id,ProvinceID,NOC_GroupingID
375,1,10.0,20.0
174,2,13.0,32.0
159,3,59.0,15.0
317,4,12.0,12.0
399,5,46.0,48.0
...,...,...,...
527,550,11.0,47.0
290,551,35.0,17.0
297,552,24.0,15.0
20,553,13.0,2.0


In [42]:
unique_ids_dicts = (
    Unique_Ids_df
    .set_index('unique_id')[['ProvinceID', 'NOC_GroupingID']]
    .to_dict(orient='index')
)

unique_ids_dicts

{1: {'ProvinceID': 10.0, 'NOC_GroupingID': 20.0},
 2: {'ProvinceID': 13.0, 'NOC_GroupingID': 32.0},
 3: {'ProvinceID': 59.0, 'NOC_GroupingID': 15.0},
 4: {'ProvinceID': 12.0, 'NOC_GroupingID': 12.0},
 5: {'ProvinceID': 46.0, 'NOC_GroupingID': 48.0},
 6: {'ProvinceID': 24.0, 'NOC_GroupingID': 20.0},
 7: {'ProvinceID': 46.0, 'NOC_GroupingID': 4.0},
 8: {'ProvinceID': 12.0, 'NOC_GroupingID': 13.0},
 9: {'ProvinceID': 13.0, 'NOC_GroupingID': 34.0},
 10: {'ProvinceID': 47.0, 'NOC_GroupingID': 4.0},
 11: {'ProvinceID': 46.0, 'NOC_GroupingID': 2.0},
 12: {'ProvinceID': 35.0, 'NOC_GroupingID': 54.0},
 13: {'ProvinceID': 12.0, 'NOC_GroupingID': 18.0},
 14: {'ProvinceID': 48.0, 'NOC_GroupingID': 40.0},
 15: {'ProvinceID': 47.0, 'NOC_GroupingID': 21.0},
 16: {'ProvinceID': 24.0, 'NOC_GroupingID': 47.0},
 17: {'ProvinceID': 48.0, 'NOC_GroupingID': 23.0},
 18: {'ProvinceID': 46.0, 'NOC_GroupingID': 30.0},
 19: {'ProvinceID': 47.0, 'NOC_GroupingID': 25.0},
 20: {'ProvinceID': 46.0, 'NOC_GroupingID':

Generating Decompositions

In [47]:
# Takes a key like 10,1 (province, NOC group), filters the polars dataframe down to just that combination 
# converts to pandas, parses the dates, sorts
# returns a clean 2-column table for that series 
def Get_Sample(unique_id: int):
    try:

        Sample = df.filter(pl.col('unique_id')==unique_id).select(['datestamp','y']).to_pandas()
        Sample['datestamp'] = pd.to_datetime(Sample['datestamp'])
        Sample = Sample.sort_values(by='datestamp', ascending=True)

        return Sample
    
    except Exception as e:
        print(f"Get_Sample(Key={unique_id}) => {e}")
        return pd.DataFrame()

# real world data can have missing months; MSTL needs a complete unbroken series to work 

# it fills gaps using 3 stratergies: 
# linear interpolation : if a value is missing between two known values, estimate it by drawing a straight line between them 
# forward fill: if there are gaps at the very start, copy the first known value backward
# backward fill:  if there are gaps at the very end, copy the last known value forward
def Process_Sample(Sample: pd.DataFrame):
    try:

        #Interpolate linearly to fill nulls
        Sample['y'] = Sample['y'].interpolate(method='linear')

        #Forward fill remaining NaNs at the start of the series
        Sample['y'] = Sample['y'].fillna(method='ffill')

        #Backward fill remaining NaNs at the end of the series
        Sample['y'] = Sample['y'].fillna(method='bfill')

        return Sample

    except Exception as e:
        print(f"Process_Sample() => {e}")
        return pd.DataFrame()

# sets the date column as the index so MSTL receives a proper time series object
def Convert_Sample(Sample: pd.DataFrame):
    try:

        SampleTS = Sample.sort_values('datestamp')['y']
        SampleTS.index = Sample['datestamp']
        return SampleTS
    

    except Exception as e:
        print(f"Convert_Sample() => {e}")
        return pd.DataFrame()

# Just wraps the MSTL(periods=[12, 6, 3]).fit() call in a safe try/except
# Returns the result, or None if it fails
def Run_Decomposition(SampleTS: pd.Series):
    try:
        mstl = MSTL(SampleTS, periods=[12, 6, 3])
        result = mstl.fit()

        return result

    except Exception as e:
        print(f"Run_Decompostion => {e}")
        return None
# extracts observed, trend, seasonalities, residual and joins them into one clean DataFrame
# Adds the ds column and resets the index
# Returns the finished decomposition table for one series

def Consolidate_Results(result):
    try:
        observed = result.observed
        trend = result.trend
        seasonalities = result.seasonal
        resid = result.resid

        df_decomp = (
            observed.rename("observed")
            .to_frame()
            .join(trend.rename("trend"))
            .join(resid.rename("resid"))
            .join(seasonalities)  
        )

        

        df_decomp['ds'] = df_decomp.index.values
        df_decomp = df_decomp.reset_index(drop=True)


        return df_decomp

    except Exception as e:
        print(f"Consolidate_Results() => {e}")
        return pd.DataFrame()
# Starts looping over every key with a tqdm progress bar

def main(unique_ids):
    try:
        All_Decompostions = []
        for unique_id in tqdm(unique_ids, desc='Processing Series', unit='Keys'):
            provinceid = unique_ids_dicts[unique_id]['ProvinceID']
            noc_groupingid = unique_ids_dicts[unique_id]['NOC_GroupingID']

            Sample = Get_Sample(unique_id=unique_id) # fetch the raw series ; skip if empty


            if len(Sample) == 0:
                print(f"Skipping {unique_id}")
                continue

            Sample = Process_Sample(Sample=Sample) # fill any missing values 


            if len(Sample) == 0:
                print(f"Skipping {unique_id}")
                continue

            SampleTS = Convert_Sample(Sample=Sample) #format as time series 


            if len(SampleTS) == 0:
                print(f"Skipping {unique_id}")
                continue

            result = Run_Decomposition(SampleTS=SampleTS) # run MSTL 


            try:
                df_decomp = Consolidate_Results(result=result) # package the output
                df_decomp['unique_id'] = unique_id
                df_decomp['provinceid'] = provinceid
                df_decomp['noc_groupingid'] = noc_groupingid

                
                All_Decompostions.append(df_decomp)
            except Exception as e:
                print(f"Skipping {unique_id} => {e}")
            
            
        
        df_All_Decompositions = pd.concat(All_Decompostions, axis=0)
        return df_All_Decompositions

    except Exception as e:
        return pd.DataFrame()

In [48]:
warnings.filterwarnings('ignore')
unique_ids = list(unique_ids_dicts.keys())
df_All_Decompositions = main(unique_ids=unique_ids)

Processing Series: 100%|██████████| 554/554 [00:08<00:00, 64.85Keys/s]


In [49]:
df_All_Decompositions

,observed,trend,resid,seasonal_3,seasonal_6,seasonal_12,ds,unique_id,provinceid,noc_groupingid
0,12.9,12.347674,0.489862,0.232237,0.079952,-0.249724,2000-01-01,1,10.0,20.0
1,14.2,12.355618,1.326067,0.087461,0.262641,0.168213,2000-02-01,1,10.0,20.0
2,12.0,12.371252,0.275493,-0.292395,-0.254699,-0.099651,2000-03-01,1,10.0,20.0
3,12.2,12.393499,-0.150567,0.193259,-0.084712,-0.151478,2000-04-01,1,10.0,20.0
4,11.8,12.420483,-0.928397,0.084314,-0.344686,0.568286,2000-05-01,1,10.0,20.0
...,...,...,...,...,...,...,...,...,...,...
304,18.9,19.964805,-2.208556,-0.469153,0.006881,1.606023,2025-05-01,554,24.0,23.0
305,21.8,20.446673,1.719564,0.588108,-0.594700,-0.359644,2025-06-01,554,24.0,23.0
306,22.3,20.963003,0.838292,-0.119591,0.666641,-0.048345,2025-07-01,554,24.0,23.0
307,24.5,21.518192,4.056362,-0.555319,0.078180,-0.597416,2025-08-01,554,24.0,23.0


In [51]:
Sample = df_All_Decompositions[(df_All_Decompositions['provinceid']==35) & (df_All_Decompositions['noc_groupingid']==18)]
Sample

,observed,trend,resid,seasonal_3,seasonal_6,seasonal_12,ds,unique_id,provinceid,noc_groupingid
0,66.9,69.593255,0.537614,-1.746469,-1.562143,0.077743,2000-01-01,457,35.0,18.0
1,71.5,69.765834,-0.531222,0.182931,0.138196,1.944261,2000-02-01,457,35.0,18.0
2,78.1,69.953260,4.448634,1.398215,2.399446,-0.099556,2000-03-01,457,35.0,18.0
3,68.5,70.156885,1.065773,-1.556273,1.914154,-3.080539,2000-04-01,457,35.0,18.0
4,64.7,70.374648,-1.521208,0.264579,-0.387220,-4.030798,2000-05-01,457,35.0,18.0
...,...,...,...,...,...,...,...,...,...,...
304,98.1,106.931901,-7.435525,-2.351277,4.079785,-3.124883,2025-05-01,457,35.0,18.0
305,98.1,106.127545,-6.124126,-1.523521,0.407496,-0.787393,2025-06-01,457,35.0,18.0
306,110.0,105.304168,4.738475,4.338528,-1.819512,-2.561658,2025-07-01,457,35.0,18.0
307,105.5,104.482290,8.810961,-2.814854,-3.619645,-1.358752,2025-08-01,457,35.0,18.0


In [52]:
# creates a 6-panel stacked chart, all sharing the time axis on the x-axis
# each panel gets one component of the decomposition 

fig = make_subplots(
    rows=6,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.05,
    subplot_titles=[
        "Employment in Thousands",
        "Trend",
        "Annual Seasonality",
        "Semi-Annual Seasonality",
        "Quarterly Seasonality",
        "Residuals"
        
    ]
)

# Observed: solid line; raw data - shown as-is
fig.add_trace(
    go.Scatter(
        x=Sample['ds'],
        y=Sample['observed'],
        mode="lines",
        name="Employment in Thousands",
        line=dict(color='#13256B', width=2.5)
    ),
    row=1,
    col=1
)

# Trend: dashed line; same color as observed but dashed to signal its a smoothed version
fig.add_trace(
    go.Scatter(
        x=Sample['ds'],
        y=Sample['trend'],
        mode="lines",
        name="Trend",
        line=dict(color='#13256B', width=2.5, dash='dash')
    ),
    row=2,
    col=1
)

# Seasonal_12: solid line ; annual cycle
fig.add_trace(
    go.Scatter(
        x=Sample['ds'],
        y=Sample['seasonal_12'],
        mode="lines",
        name="Annual Seasonality",
        line=dict(color='#064241', width=2.5)
    ),
    row=3,
    col=1
)

# Seasonal_6: semi annual
fig.add_trace(
    go.Scatter(
        x=Sample['ds'],
        y=Sample['seasonal_6'],
        mode="lines",
        name="Semi-Annual Seasonality",
        line=dict(color='#17615E', width=2.5)
    ),
    row=4,
    col=1
)

#Seasonal_3: quarterly cycle
fig.add_trace(
    go.Scatter(
        x=Sample['ds'],
        y=Sample['seasonal_3'],
        mode="lines",
        name="Quarterly Seasonality",
        line=dict(color="#42B3AF", width=2.5)
    ),
    row=5,
    col=1
)

# Residuals: dots; Residuals are shown as scatter dots — not a line — because they're random noise with no meaningful connection between points
fig.add_trace(
    go.Scatter(
        x=Sample['ds'],
        y=Sample['resid'],
        mode="markers",
        name="Residuals",
        marker=dict(color="#580A0A", size=8)
    ),
    row=6,
    col=1
)



fig.update_layout(
    height=900,
    width=1500,
    title="Multi Seasonal Decomposition of Professional occupations in engineering [213] in Ontario",
    showlegend=False
)
fig.update_layout(template='ggplot2')
fig.show()

In [50]:
df_All_Decompositions.to_parquet('Data/SeasonalDecomposedData.parquet',index=False)